# Experiments 7 — Explicit Text-to-Structure Fidelity

Purpose: evaluate whether LLM shared decomposition recovers **only explicitly stated** physical parts, multiplicities, root, and direct attachment relations. No 3D geometry is generated here.

Benchmark: 30 base structures × 3 paraphrases = 90 prompts. DeepSeek and Qwen use the same extraction prompt. Retries receive schema errors only; GT semantic errors are never fed back.

In [2]:
import os
import importlib
import pandas as pd
import methods
from IPython.display import display

importlib.reload(methods)
from methods import *

In [3]:
e7 = Experiments7Config()
e7.project_dir = "."
e7.output_dir = "experiments7_results"
e7.benchmark_path = "explicit_structure_benchmark_90.json"
e7.temperature = 0.0
e7.max_tokens = 4096
e7.max_attempts = 3
e7.workers = 4
e7.bootstrap_samples = 10000
e7.statistics_seed = 42
e7.allow_api = True

deepseek7 = Experiments7ModelConfig(
    label="deepseek",
    model=os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),
    base_url="https://api.deepseek.com",
    api_key_env="DEEPSEEK_API_KEY",
    extra_body={},
)

qwen7 = Experiments7ModelConfig(
    label="qwen3_7_plus",
    model="qwen3.7-plus",
    base_url="https://maas.qianwenaiapi.com/compatible-mode/v1",
    api_key_env="sk-ws-H.PPIPXHL.v7xV.MEUCIAHrO3UsiMeVOtF4a28BpY27WF32q4y1WiKkdTZLmFz6AiEAuoWeP5FZIVTx1XM8E94tWmyNTsfWkJfTtoM6AT2DIac",
    extra_body={"enable_thinking": False},
)

model_configs = [deepseek7, qwen7]

print("output:", e7.output_path())
print([(m.label, m.model) for m in model_configs])

output: D:\MyJupyter\Works\ASPT\experiments7_results
[('deepseek', 'deepseek-chat'), ('qwen3_7_plus', 'qwen3.7-plus')]


In [5]:
benchmark7 = load_experiments7_benchmark(
    e7,
    model_configs,
)

frame7 = pd.DataFrame(benchmark7["objects"])

print("prompts:", len(frame7))
print("base structures:", frame7["base_structure_id"].nunique())

display(
    frame7.groupby("category").size().rename("prompts").to_frame()
)

display(
    frame7.groupby("complexity").size().rename("prompts").to_frame()
)

display(
    frame7[[
        "object_id",
        "category",
        "complexity",
        "paraphrase_style",
        "description",
    ]].head(6)
)

prompts: 90
base structures: 30


,prompts
category,
Appliances,15
Electronics,15
Furniture,15
MechanicalComposite,15
Tools,15
Vehicles,15


,prompts
complexity,
complex,42
medium,30
simple,18


,object_id,category,complexity,paraphrase_style,description
0,E7_FUR01_P1,Furniture,simple,inventory_first,A four-legged stool consists of one seat and 4...
1,E7_FUR01_P2,Furniture,simple,integrated_relations,This four-legged stool uses exactly one seat a...
2,E7_FUR01_P3,Furniture,simple,assembly_instruction,Build the four-legged stool from exactly one s...
3,E7_FUR02_P1,Furniture,medium,inventory_first,"A writing desk consists of one tabletop, 4 leg..."
4,E7_FUR02_P2,Furniture,medium,integrated_relations,"This writing desk uses exactly one tabletop, 4..."
5,E7_FUR02_P3,Furniture,medium,assembly_instruction,Build the writing desk from exactly one tablet...


In [6]:
deepseek_client7 = create_openai_compatible_client(
    api_key_env=deepseek7.api_key_env,
    base_url=deepseek7.base_url,
    timeout=e7.timeout,
)

qwen_client7 = create_openai_compatible_client(
    api_key_env=qwen7.api_key_env,
    base_url=qwen7.base_url,
    timeout=e7.timeout,
)

print("clients ready")

sk-ws-H.PPIPXHL.v7xV.MEUCIAHrO3UsiMeVOtF4a28BpY27WF32q4y1WiKkdTZLmFz6AiEAuoWeP5FZIVTx1XM8E94tWmyNTsfWkJfTtoM6AT2DIac（隐藏输入，仅保存在当前进程）： ········


clients ready


In [7]:
deepseek_generations7 = run_experiments7_decomposition(
    benchmark7,
    e7,
    deepseek7,
    client=deepseek_client7,
)

print(
    "DeepSeek valid:",
    deepseek_generations7["valid_count"],
    "/",
    deepseek_generations7["prompt_count"],
)

deepseek: 15/90 processed, valid=15
deepseek: 30/90 processed, valid=30
deepseek: 45/90 processed, valid=45
deepseek: 60/90 processed, valid=60
deepseek: 75/90 processed, valid=75
deepseek: 90/90 processed, valid=90
DeepSeek valid: 90 / 90


In [8]:
qwen_generations7 = run_experiments7_decomposition(
    benchmark7,
    e7,
    qwen7,
    client=qwen_client7,
)

print(
    "Qwen valid:",
    qwen_generations7["valid_count"],
    "/",
    qwen_generations7["prompt_count"],
)

qwen3_7_plus: 15/90 processed, valid=15
qwen3_7_plus: 30/90 processed, valid=30
qwen3_7_plus: 45/90 processed, valid=45
qwen3_7_plus: 60/90 processed, valid=60
qwen3_7_plus: 75/90 processed, valid=75
qwen3_7_plus: 90/90 processed, valid=90
Qwen valid: 90 / 90


In [9]:
e7.allow_api = False

print("API disabled for all remaining cells")

API disabled for all remaining cells


In [10]:
generations7 = {
    deepseek7.label: deepseek_generations7,
    qwen7.label: qwen_generations7,
}

evaluation7 = run_experiments7_evaluation(
    benchmark7,
    generations7,
    e7,
)

results7 = summarize_experiments7_results(
    benchmark7,
    evaluation7,
    e7,
)

display(
    experiments7_summary_view(results7)[[
        "schema_valid",
        "part_instance_f1",
        "type_count_accuracy",
        "connection_f1",
        "root_correct",
        "exact_inventory",
        "exact_connections",
        "exact_structure",
        "unsupported_part_instances",
    ]]
)

print("cross-model exact agreement")
display(pd.DataFrame([results7["cross_model_exact"]]))

metric,schema_valid,part_instance_f1,type_count_accuracy,connection_f1,root_correct,exact_inventory,exact_connections,exact_structure,unsupported_part_instances
model,,,,,,,,,
deepseek,1.0,1.0000,1.0000,1.0000,1.0,1.0000,1.0000,1.0000,0.0000
qwen3_7_plus,1.0,0.9951,0.9968,0.9944,1.0,0.9889,0.9889,0.9889,0.0444


cross-model exact agreement


,model_a,model_b,n,both_exact,a_only_exact,b_only_exact,neither_exact,exact_agreement_rate,mcnemar_exact_p
0,deepseek,qwen3_7_plus,90,89,1,0,0,0.988889,1.0


In [11]:
print("Paraphrase robustness by model")
base7 = pd.DataFrame(results7["base_structure_consistency"])

display(
    base7.groupby("model").agg(
        all_three_exact_rate=("all_three_exact", "mean"),
        mean_exact_paraphrases=("exact_structure_count", "mean"),
        mean_part_f1=("mean_part_f1", "mean"),
        mean_connection_f1=("mean_connection_f1", "mean"),
    ).round(4)
)

print("By complexity")
complex7 = pd.DataFrame(results7["complexity_summary"])

display(
    complex7[complex7["metric"].isin([
        "part_instance_f1",
        "connection_f1",
        "exact_structure",
    ])].pivot_table(
        index=["model", "complexity"],
        columns="metric",
        values="mean",
    ).round(4)
)

print("By category")
category7 = pd.DataFrame(results7["category_summary"])

display(
    category7[category7["metric"].isin([
        "part_instance_f1",
        "connection_f1",
        "exact_structure",
    ])].pivot_table(
        index=["model", "category"],
        columns="metric",
        values="mean",
    ).round(4)
)

Paraphrase robustness by model


,all_three_exact_rate,mean_exact_paraphrases,mean_part_f1,mean_connection_f1
model,,,,
deepseek,1.0000,3.0000,1.0000,1.0000
qwen3_7_plus,0.9667,2.9667,0.9951,0.9944


By complexity


metric                   connection_f1  exact_structure  part_instance_f1
model        complexity                                                  
deepseek     complex            1.0000           1.0000            1.0000
             medium             1.0000           1.0000            1.0000
             simple             1.0000           1.0000            1.0000
qwen3_7_plus complex            0.9881           0.9762            0.9894
             medium             1.0000           1.0000            1.0000
             simple             1.0000           1.0000            1.0000

By category


metric                            connection_f1  exact_structure  \
model        category                                              
deepseek     Appliances                  1.0000           1.0000   
             Electronics                 1.0000           1.0000   
             Furniture                   1.0000           1.0000   
             MechanicalComposite         1.0000           1.0000   
             Tools                       1.0000           1.0000   
             Vehicles                    1.0000           1.0000   
qwen3_7_plus Appliances                  0.9667           0.9333   
             Electronics                 1.0000           1.0000   
             Furniture                   1.0000           1.0000   
             MechanicalComposite         1.0000           1.0000   
             Tools                       1.0000           1.0000   
             Vehicles                    1.0000           1.0000   

metric                            part_instance_f1  
model        category                               
deepseek     Appliances                     1.0000  
             Electronics                    1.0000  
             Furniture                      1.0000  
             MechanicalComposite            1.0000  
             Tools                          1.0000  
             Vehicles                       1.0000  
qwen3_7_plus Appliances                     0.9704  
             Electronics                    1.0000  
             Furniture                      1.0000  
             MechanicalComposite            1.0000  
             Tools                          1.0000  
             Vehicles                       1.0000

In [12]:
for label, rows in evaluation7["models"].items():
    failures = pd.DataFrame(rows)
    failures = failures[
        failures["exact_structure"] == 0
    ].copy()

    print()
    print(label, "non-exact prompts:", len(failures))

    if not failures.empty:
        display(
            failures[[
                "object_id",
                "category",
                "complexity",
                "schema_valid",
                "part_instance_f1",
                "connection_f1",
                "root_correct",
                "missing_part_instances",
                "unsupported_part_instances",
                "missing_connections",
                "unsupported_connections",
            ]].head(20)
        )


deepseek non-exact prompts: 0

qwen3_7_plus non-exact prompts: 1


,object_id,category,complexity,schema_valid,part_instance_f1,connection_f1,root_correct,missing_part_instances,unsupported_part_instances,missing_connections,unsupported_connections
71,E7_APP04_P3,Appliances,complex,1,0.555556,0.5,1,4,4,4,4


In [13]:
review7 = export_experiments7_review(
    e7,
    model_configs,
)

print("upload:", review7.resolve())

upload: D:\MyJupyter\Works\ASPT\experiments7_results\experiments7_review.zip
